# Phase 6 — Real Database Design

## Objective

Design a persistent database for storing missing-person records,
original photographs, and searchable face embeddings.

## Missing Person Record

Each missing-person record will contain:

- Person ID
- Name
- Age
- Gender
- Last seen location
- Contact information
- Report date
- Current status
- Original photographs
- Face embeddings

## Database Requirement

The database should allow the system to:

1. Register a missing person.
2. Store their personal information.
3. Store one or more original photographs.
4. Generate and store face embeddings.
5. Search stored embeddings for potential matches.
6. Return candidate information for human verification.

## Database Tables

The database will use three main tables.

### 1. Persons

Stores missing-person information.

Fields:

- person_id
- name
- age
- gender
- last_seen_location
- contact_information
- report_date
- status

### 2. Photos

Stores references to the original photographs.

Fields:

- photo_id
- person_id
- file_path
- uploaded_at

Relationship:

One person can have multiple photographs.

### 3. Embeddings

Stores the facial representation generated from a photograph.

Fields:

- embedding_id
- photo_id
- embedding
- model_name
- created_at

Relationship:

Each photograph can have a corresponding face embedding.

## Relationship

Persons 1 ──── * Photos 1 ──── 1 Embeddings

In [1]:
import sqlite3

DATABASE_PATH = "missing_persons.db"

connection = sqlite3.connect(DATABASE_PATH)

print("Database connection successful!")

Database connection successful!


In [13]:
import os
import cv2
import numpy as np
from deepface import DeepFace

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS persons (
    person_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    age INTEGER,
    gender TEXT,
    last_seen_location TEXT,
    contact_information TEXT,
    report_date TEXT,
    status TEXT
)
""")

connection.commit()

print("Persons table created successfully!")

Persons table created successfully!


In [3]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS photos (
    photo_id INTEGER PRIMARY KEY AUTOINCREMENT,
    person_id INTEGER NOT NULL,
    file_path TEXT NOT NULL,
    uploaded_at TEXT,
    FOREIGN KEY (person_id) REFERENCES persons(person_id)
)
""")

connection.commit()

print("Photos table created successfully!")

Photos table created successfully!


In [4]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS embeddings (
    embedding_id INTEGER PRIMARY KEY AUTOINCREMENT,
    photo_id INTEGER NOT NULL,
    embedding BLOB NOT NULL,
    model_name TEXT NOT NULL,
    created_at TEXT,
    FOREIGN KEY (photo_id) REFERENCES photos(photo_id)
)
""")

connection.commit()

print("Embeddings table created successfully!")

Embeddings table created successfully!


In [5]:
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""")

tables = cursor.fetchall()

print("Database tables:")

for table in tables:
    print("-", table[0])

Database tables:
- embeddings
- persons
- photos
- sqlite_sequence


In [6]:
cursor.execute("""
INSERT INTO persons (
    name,
    age,
    gender,
    last_seen_location,
    contact_information,
    report_date,
    status
)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", (
    "Test Person",
    25,
    "Male",
    "Delhi",
    "test@example.com",
    "2026-09-10",
    "MISSING"
))

connection.commit()

print("Test person inserted successfully!")

Test person inserted successfully!


In [7]:
cursor.execute("""
SELECT *
FROM persons
""")

people = cursor.fetchall()

for person in people:
    print(person)

(1, 'Test Person', 25, 'Male', 'Delhi', 'test@example.com', '2026-09-10', 'MISSING')


In [8]:
cursor.execute("""
INSERT INTO photos (
    person_id,
    file_path,
    uploaded_at
)
VALUES (?, ?, ?)
""", (
    1,
    "deepface_repo/tests/unit/dataset/img1.jpg",
    "2026-09-10"
))

connection.commit()

print("Photo record inserted successfully!")

Photo record inserted successfully!


In [15]:
def detect_faces(image_path):

    detections = DeepFace.extract_faces(
        img_path=image_path,
        detector_backend="mtcnn",
        enforce_detection=False,
        align=True
    )

    valid_faces = [
        detection
        for detection in detections
        if detection["confidence"] > 0
    ]

    return valid_faces


print("Face detection function ready!")

Face detection function ready!


In [16]:
def generate_face_embeddings(image_path):

    faces = detect_faces(image_path)

    embeddings = []

    for face in faces:

        face_crop = face["face"]

        embedding = DeepFace.represent(
            img_path=face_crop,
            model_name="ArcFace",
            detector_backend="skip",
            enforce_detection=False
        )[0]["embedding"]

        embeddings.append(np.array(embedding))

    return embeddings


print("Face embedding function ready!")

Face embedding function ready!


In [17]:
photo_path = "deepface_repo/tests/unit/dataset/img1.jpg"

embeddings = generate_face_embeddings(photo_path)

print("Number of embeddings:", len(embeddings))
print("Embedding shape:", embeddings[0].shape)

Number of embeddings: 1
Embedding shape: (512,)


In [18]:
# Get the photo_id of the photo we inserted earlier

cursor.execute("""
SELECT photo_id
FROM photos
WHERE person_id = ?
ORDER BY photo_id DESC
LIMIT 1
""", (1,))

photo_id = cursor.fetchone()[0]

print("Photo ID:", photo_id)

Photo ID: 1


In [19]:
# Store the ArcFace embedding in the database

embedding = embeddings[0]

cursor.execute("""
INSERT INTO embeddings (
    photo_id,
    embedding,
    model_name,
    created_at
)
VALUES (?, ?, ?, ?)
""", (
    photo_id,
    embedding.tobytes(),
    "ArcFace",
    "2026-09-10"
))

connection.commit()

print("Embedding stored successfully!")

Embedding stored successfully!


In [20]:
# Verify the embedding stored in the database

cursor.execute("""
SELECT embedding_id, photo_id, model_name, length(embedding)
FROM embeddings
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 1, 'ArcFace', 4096)


In [21]:
# Read the stored embedding back from the database

cursor.execute("""
SELECT embedding
FROM embeddings
WHERE photo_id = ?
""", (photo_id,))

stored_blob = cursor.fetchone()[0]

stored_embedding = np.frombuffer(
    stored_blob,
    dtype=np.float64
)

print("Stored embedding shape:", stored_embedding.shape)
print("Stored embedding type:", stored_embedding.dtype)

Stored embedding shape: (512,)
Stored embedding type: float64


In [22]:
# Check that the stored embedding is identical to the original

difference = np.max(
    np.abs(embedding - stored_embedding)
)

print("Maximum difference:", difference)

Maximum difference: 0.0


In [23]:
# Load all stored embeddings from the database

cursor.execute("""
SELECT
    persons.person_id,
    persons.name,
    photos.photo_id,
    photos.file_path,
    embeddings.embedding,
    embeddings.model_name
FROM embeddings
JOIN photos
    ON embeddings.photo_id = photos.photo_id
JOIN persons
    ON photos.person_id = persons.person_id
""")

database_records = cursor.fetchall()

print("Number of embedding records:", len(database_records))

for record in database_records:
    print(
        "Person:", record[1],
        "| Photo ID:", record[2],
        "| Model:", record[5]
    )

Number of embedding records: 1
Person: Test Person | Photo ID: 1 | Model: ArcFace


In [24]:
# Convert database BLOBs into NumPy embeddings

search_database = []

for record in database_records:

    person_id = record[0]
    name = record[1]
    photo_id = record[2]
    file_path = record[3]
    embedding_blob = record[4]
    model_name = record[5]

    embedding_vector = np.frombuffer(
        embedding_blob,
        dtype=np.float64
    )

    search_database.append({
        "person_id": person_id,
        "name": name,
        "photo_id": photo_id,
        "file_path": file_path,
        "embedding": embedding_vector,
        "model_name": model_name
    })

print("Search database created!")
print("Number of records:", len(search_database))
print("Embedding shape:", search_database[0]["embedding"].shape)

Search database created!
Number of records: 1
Embedding shape: (512,)


In [25]:
# Calculate cosine distance between two face embeddings

def cosine_distance(embedding_a, embedding_b):

    similarity = np.dot(embedding_a, embedding_b) / (
        np.linalg.norm(embedding_a) *
        np.linalg.norm(embedding_b)
    )

    distance = 1 - similarity

    return distance


print("Cosine distance function ready!")

Cosine distance function ready!


In [26]:
# Generate embedding for the query photograph

query_path = "deepface_repo/tests/unit/dataset/img1.jpg"

query_embeddings = generate_face_embeddings(query_path)

print("Number of query embeddings:", len(query_embeddings))
print("Query embedding shape:", query_embeddings[0].shape)

Number of query embeddings: 1
Query embedding shape: (512,)


In [27]:
# Compare query embedding with stored database embeddings

query_embedding = query_embeddings[0]

for record in search_database:

    database_embedding = record["embedding"]

    distance = cosine_distance(
        query_embedding,
        database_embedding
    )

    print("Person:", record["name"])
    print("Photo ID:", record["photo_id"])
    print("Cosine distance:", round(distance, 4))

Person: Test Person
Photo ID: 1
Cosine distance: 0.0


In [28]:
def search_database_by_embedding(query_embedding, top_k=5):

    results = []

    for record in search_database:

        database_embedding = record["embedding"]

        distance = cosine_distance(
            query_embedding,
            database_embedding
        )

        results.append({
            "person_id": record["person_id"],
            "name": record["name"],
            "photo_id": record["photo_id"],
            "file_path": record["file_path"],
            "distance": distance
        })

    results.sort(key=lambda x: x["distance"])

    return results[:top_k]


print("Database search function ready!")

Database search function ready!


In [29]:
# Search the database using the query embedding

search_results = search_database_by_embedding(
    query_embedding,
    top_k=5
)

for rank, result in enumerate(search_results, start=1):

    print(
        f"{rank}. {result['name']} "
        f"| Photo ID: {result['photo_id']} "
        f"| Distance: {result['distance']:.4f}"
    )

1. Test Person | Photo ID: 1 | Distance: 0.0000


In [30]:
def register_person(name, age, gender, last_seen_location,
                    contact_information, report_date, status,
                    photo_path):

    # Insert person
    cursor.execute("""
    INSERT INTO persons (
        name,
        age,
        gender,
        last_seen_location,
        contact_information,
        report_date,
        status
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        name,
        age,
        gender,
        last_seen_location,
        contact_information,
        report_date,
        status
    ))

    person_id = cursor.lastrowid

    # Generate face embedding
    face_embeddings = generate_face_embeddings(photo_path)

    if len(face_embeddings) != 1:
        connection.rollback()
        return "ERROR: Expected exactly one face."

    # Insert photo
    cursor.execute("""
    INSERT INTO photos (
        person_id,
        file_path,
        uploaded_at
    )
    VALUES (?, ?, ?)
    """, (
        person_id,
        photo_path,
        report_date
    ))

    photo_id = cursor.lastrowid

    # Store embedding
    embedding = face_embeddings[0]

    cursor.execute("""
    INSERT INTO embeddings (
        photo_id,
        embedding,
        model_name,
        created_at
    )
    VALUES (?, ?, ?, ?)
    """, (
        photo_id,
        embedding.tobytes(),
        "ArcFace",
        report_date
    ))

    connection.commit()

    return "SUCCESS"

In [31]:
result = register_person(
    name="Test Person 2",
    age=30,
    gender="Male",
    last_seen_location="Mumbai",
    contact_information="test2@example.com",
    report_date="2026-09-10",
    status="MISSING",
    photo_path="deepface_repo/tests/unit/dataset/img3.jpg"
)

print(result)

SUCCESS


In [32]:
# Verify persons, photos, and embeddings

cursor.execute("SELECT COUNT(*) FROM persons")
person_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM photos")
photo_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM embeddings")
embedding_count = cursor.fetchone()[0]

print("Persons:", person_count)
print("Photos:", photo_count)
print("Embeddings:", embedding_count)

Persons: 2
Photos: 2
Embeddings: 2


In [34]:
# Reload all embeddings from SQLite after registration

cursor.execute("""
SELECT
    persons.person_id,
    persons.name,
    photos.photo_id,
    photos.file_path,
    embeddings.embedding,
    embeddings.model_name
FROM embeddings
JOIN photos
    ON embeddings.photo_id = photos.photo_id
JOIN persons
    ON photos.person_id = persons.person_id
""")

database_records = cursor.fetchall()

search_database = []

for record in database_records:

    search_database.append({
        "person_id": record[0],
        "name": record[1],
        "photo_id": record[2],
        "file_path": record[3],
        "embedding": np.frombuffer(
            record[4],
            dtype=np.float64
        ),
        "model_name": record[5]
    })

print("Search database reloaded!")
print("Number of records:", len(search_database))

for record in search_database:
    print(
        record["person_id"],
        "|",
        record["name"],
        "| Photo:",
        record["photo_id"],
        "| Embedding:",
        record["embedding"].shape
    )

Search database reloaded!
Number of records: 2
1 | Test Person | Photo: 1 | Embedding: (512,)
2 | Test Person 2 | Photo: 2 | Embedding: (512,)


In [35]:
# Perform a one-to-many search using the SQLite database

query_embedding = query_embeddings[0]

search_results = search_database_by_embedding(
    query_embedding,
    top_k=5
)

print("Ranked Search Results:\n")

for rank, result in enumerate(search_results, start=1):

    print(
        f"{rank}. {result['name']} "
        f"| Photo ID: {result['photo_id']} "
        f"| Distance: {result['distance']:.4f}"
    )

Ranked Search Results:

1. Test Person | Photo ID: 1 | Distance: 0.0000
2. Test Person 2 | Photo ID: 2 | Distance: 0.9516


In [36]:
THRESHOLD = 0.68

best_result = search_results[0]

if best_result["distance"] < THRESHOLD:
    status = "POTENTIAL_MATCH"
else:
    status = "NO_RELIABLE_MATCH"

print("Search Status:", status)
print("Best Candidate:", best_result["name"])
print("Best Distance:", round(best_result["distance"], 4))
print("Threshold:", THRESHOLD)

Search Status: POTENTIAL_MATCH
Best Candidate: Test Person
Best Distance: 0.0
Threshold: 0.68


In [37]:
# Test the search with a different person's photograph

query_path_2 = "deepface_repo/tests/unit/dataset/img3.jpg"

query_embeddings_2 = generate_face_embeddings(query_path_2)

query_embedding_2 = query_embeddings_2[0]

search_results_2 = search_database_by_embedding(
    query_embedding_2,
    top_k=5
)

print("Ranked Search Results:\n")

for rank, result in enumerate(search_results_2, start=1):

    print(
        f"{rank}. {result['name']} "
        f"| Photo ID: {result['photo_id']} "
        f"| Distance: {result['distance']:.4f}"
    )

Ranked Search Results:

1. Test Person 2 | Photo ID: 2 | Distance: 0.0000
2. Test Person | Photo ID: 1 | Distance: 0.9516


In [38]:
def add_photo_to_person(person_id, photo_path, uploaded_at):

    # Check that the person exists
    cursor.execute("""
    SELECT person_id
    FROM persons
    WHERE person_id = ?
    """, (person_id,))

    person = cursor.fetchone()

    if person is None:
        return "ERROR: Person not found."

    # Generate face embedding
    face_embeddings = generate_face_embeddings(photo_path)

    if len(face_embeddings) != 1:
        return "ERROR: Expected exactly one face."

    # Insert photo
    cursor.execute("""
    INSERT INTO photos (
        person_id,
        file_path,
        uploaded_at
    )
    VALUES (?, ?, ?)
    """, (
        person_id,
        photo_path,
        uploaded_at
    ))

    photo_id = cursor.lastrowid

    # Store embedding
    embedding = face_embeddings[0]

    cursor.execute("""
    INSERT INTO embeddings (
        photo_id,
        embedding,
        model_name,
        created_at
    )
    VALUES (?, ?, ?, ?)
    """, (
        photo_id,
        embedding.tobytes(),
        "ArcFace",
        uploaded_at
    ))

    connection.commit()

    return "SUCCESS"


print("Add-photo function ready!")

Add-photo function ready!


In [39]:
result = add_photo_to_person(
    person_id=1,
    photo_path="deepface_repo/tests/unit/dataset/img2.jpg",
    uploaded_at="2026-09-10"
)

print(result)

SUCCESS


In [40]:
# Verify Test Person's photos and embeddings

cursor.execute("""
SELECT
    persons.name,
    photos.photo_id,
    photos.file_path,
    embeddings.embedding_id,
    embeddings.model_name
FROM persons
JOIN photos
    ON persons.person_id = photos.person_id
JOIN embeddings
    ON photos.photo_id = embeddings.photo_id
WHERE persons.person_id = ?
""", (1,))

person_records = cursor.fetchall()

for record in person_records:
    print(
        "Person:", record[0],
        "| Photo ID:", record[1],
        "| File:", record[2],
        "| Embedding ID:", record[3],
        "| Model:", record[4]
    )

Person: Test Person | Photo ID: 1 | File: deepface_repo/tests/unit/dataset/img1.jpg | Embedding ID: 1 | Model: ArcFace
Person: Test Person | Photo ID: 3 | File: deepface_repo/tests/unit/dataset/img2.jpg | Embedding ID: 3 | Model: ArcFace


In [41]:
# Reload searchable embeddings from SQLite

cursor.execute("""
SELECT
    persons.person_id,
    persons.name,
    photos.photo_id,
    photos.file_path,
    embeddings.embedding,
    embeddings.model_name
FROM embeddings
JOIN photos
    ON embeddings.photo_id = photos.photo_id
JOIN persons
    ON photos.person_id = persons.person_id
""")

database_records = cursor.fetchall()

search_database = []

for record in database_records:

    search_database.append({
        "person_id": record[0],
        "name": record[1],
        "photo_id": record[2],
        "file_path": record[3],
        "embedding": np.frombuffer(
            record[4],
            dtype=np.float64
        ),
        "model_name": record[5]
    })

print("Search database reloaded!")
print("Number of embedding records:", len(search_database))

Search database reloaded!
Number of embedding records: 3


In [42]:
# Search using a different photo of Test Person

query_path_3 = "deepface_repo/tests/unit/dataset/img4.jpg"

query_embeddings_3 = generate_face_embeddings(query_path_3)

query_embedding_3 = query_embeddings_3[0]

search_results_3 = search_database_by_embedding(
    query_embedding_3,
    top_k=5
)

print("Ranked Search Results:\n")

for rank, result in enumerate(search_results_3, start=1):

    print(
        f"{rank}. {result['name']} "
        f"| Photo ID: {result['photo_id']} "
        f"| Distance: {result['distance']:.4f}"
    )

Ranked Search Results:

1. Test Person | Photo ID: 3 | Distance: 0.2426
2. Test Person | Photo ID: 1 | Distance: 0.4916
3. Test Person 2 | Photo ID: 2 | Distance: 0.9581


In [43]:
def search_people_from_database(query_embedding, top_k=5):

    person_results = {}

    for record in search_database:

        distance = cosine_distance(
            query_embedding,
            record["embedding"]
        )

        person_id = record["person_id"]

        if person_id not in person_results:

            person_results[person_id] = {
                "person_id": person_id,
                "name": record["name"],
                "best_distance": distance,
                "best_photo_id": record["photo_id"],
                "best_photo_path": record["file_path"],
                "photo_count": 1
            }

        else:

            person_results[person_id]["photo_count"] += 1

            if distance < person_results[person_id]["best_distance"]:

                person_results[person_id]["best_distance"] = distance
                person_results[person_id]["best_photo_id"] = record["photo_id"]
                person_results[person_id]["best_photo_path"] = record["file_path"]

    results = list(person_results.values())

    results.sort(
        key=lambda x: x["best_distance"]
    )

    return results[:top_k]


print("Person-level search function ready!")

Person-level search function ready!


In [44]:
# Person-level search using the unseen query image

person_search_results = search_people_from_database(
    query_embedding_3,
    top_k=5
)

print("Person-Level Search Results:\n")

for rank, result in enumerate(person_search_results, start=1):

    print(
        f"{rank}. {result['name']} "
        f"| Best Distance: {result['best_distance']:.4f} "
        f"| Best Photo ID: {result['best_photo_id']} "
        f"| Reference Photos: {result['photo_count']}"
    )

Person-Level Search Results:

1. Test Person | Best Distance: 0.2426 | Best Photo ID: 3 | Reference Photos: 2
2. Test Person 2 | Best Distance: 0.9581 | Best Photo ID: 2 | Reference Photos: 1


In [45]:
THRESHOLD = 0.68

best_person = person_search_results[0]

if best_person["best_distance"] < THRESHOLD:
    status = "POTENTIAL_MATCH"
else:
    status = "NO_RELIABLE_MATCH"

print("Search Status:", status)
print("Best Candidate:", best_person["name"])
print("Best Distance:", round(best_person["best_distance"], 4))
print("Reference Photos:", best_person["photo_count"])
print("Threshold:", THRESHOLD)

Search Status: POTENTIAL_MATCH
Best Candidate: Test Person
Best Distance: 0.2426
Reference Photos: 2
Threshold: 0.68


In [46]:
def search_missing_person_database(image_path, top_k=5):

    # Generate query embedding
    query_embeddings = generate_face_embeddings(image_path)

    # No face
    if len(query_embeddings) == 0:
        return {
            "status": "NO_FACE",
            "results": []
        }

    # Multiple faces
    if len(query_embeddings) > 1:
        return {
            "status": "MULTIPLE_FACES",
            "results": []
        }

    # Single face
    query_embedding = query_embeddings[0]

    # Search database
    results = search_people_from_database(
        query_embedding,
        top_k=top_k
    )

    if len(results) == 0:
        return {
            "status": "NO_DATABASE_RECORDS",
            "results": []
        }

    # Threshold decision
    best_result = results[0]

    if best_result["best_distance"] < THRESHOLD:
        status = "POTENTIAL_MATCH"
    else:
        status = "NO_RELIABLE_MATCH"

    return {
        "status": status,
        "results": results
    }


print("Complete database search function ready!")

Complete database search function ready!


In [47]:
# Complete end-to-end database search

result = search_missing_person_database(
    "deepface_repo/tests/unit/dataset/img4.jpg",
    top_k=5
)

print("Search Status:", result["status"])
print()

for rank, person in enumerate(result["results"], start=1):

    print(
        f"{rank}. {person['name']} "
        f"| Best Distance: {person['best_distance']:.4f} "
        f"| Reference Photos: {person['photo_count']}"
    )

Search Status: POTENTIAL_MATCH

1. Test Person | Best Distance: 0.2426 | Reference Photos: 2
2. Test Person 2 | Best Distance: 0.9581 | Reference Photos: 1


## Phase 6 Results and Conclusion

### Database Architecture

The real database prototype was implemented using SQLite.

The database contains three main tables:

- `persons` — stores missing-person information.
- `photos` — stores photograph records linked to a person.
- `embeddings` — stores ArcFace face embeddings linked to each photograph.

The relationship is:

```text
Person
   |
   +---- Photo
            |
            +---- Face Embedding